# Original matrix HHL: precision, fidelity and actual circuit resources

Preserve the original rounded 8 by 8 matrix, eight-element RHS, all seven phase
precisions 2 through 8, reciprocal rotations, physical postselection, squared
overlap and final cost/fidelity figure. Native quantum functions feed the same
FlagQuantum execution path as the independent Circuit oracle.

The source's rounded RHS has norm 0.9999999994356988. A unitary preparation
requires a unit vector; this example explicitly normalizes that unchanged rounded
vector. The classical solve keeps the original unnormalized RHS. The comparison
normalizes both solutions, so the stated squared overlap is unaffected by this
preparation normalization. No changed coefficients or hidden normalization policy
is introduced into the compiler. Optional source Qiskit cells are fully commented;
their arrays are historical observations, not new measurements.


In [ ]:
from pathlib import Path
from typing import Annotated

from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.tutorials.technology_demonstrations.hhl.hhl_example.overview.ansatz import (
    build_program,
)
from examples.synthesis.tutorials.technology_demonstrations.hhl.hhl_example.overview.physics import (
    AMPLITUDES,
    CLASSICAL,
    HISTORICAL_CX,
    HISTORICAL_DEPTHS,
    HISTORICAL_FIDELITIES,
    HISTORICAL_WIDTHS,
    MATRIX,
    PRECISIONS,
    RAW_B,
    RAW_B_NORM,
    UNITARY,
    B,
    reciprocal_angles,
)
from examples.synthesis.tutorials.technology_demonstrations.hhl.hhl_example.overview.workflow import (
    plot_study,
    verify_program,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QBit,
    QNum,
    allocate,
    apply,
    prepare_amplitudes,
    qfunc,
    qpe,
    select_rotation,
    unitary,
    within,
)

print("Original matrix:", MATRIX)
print("Original rounded RHS:", RAW_B)
print("Rounded RHS norm:", RAW_B_NORM)
print("Explicit preparation normalization:", B)
print("Classical solution of original RHS:", CLASSICAL)
assert MATRIX.shape == (8, 8) and len(AMPLITUDES) == 8
assert abs(RAW_B_NORM - 1) > 1e-10
assert abs(float(B.norm()) - 1) < 1e-14
output_directory = Path("artifacts/development/hhl_precision")


In [ ]:
print("Native costs use exact public CX/U3 conversion")
print("Source Qiskit optimization level: 1 (historical, not rerun)")


In [ ]:
@qfunc
def hhl_precision_four(
    amplitudes: list[float],
    angles: list[float],
    res: Output[Annotated[QNum, 3]],
    phase: Output[Annotated[QNum, (4, False, 4)]],
    indicator: Output[QBit],
) -> None:
    prepare_amplitudes(amplitudes, 0, res)
    allocate(phase)
    allocate(indicator)
    with within():
        qpe(lambda: unitary(UNITARY, res), phase)
    with apply():
        select_rotation("Y", angles, phase, indicator)


visible_program = synth.synthesize(
    hhl_precision_four,
    classical_inputs={
        "amplitudes": list(AMPLITUDES),
        "angles": list(reciprocal_angles(4)),
    },
    max_qubits=8,
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
visible_result = fq.run(visible_program)
print(
    "Named indicator readout:",
    dict(visible_program.probabilities(visible_result, registers=("indicator",))),
)


The fixed precision-four root above demonstrates the user flow. The module
contains seven literal fractional annotations for the complete source precision
sweep; it does not introduce dynamic annotation or generic QCallable support.
The full complex independent reference uses an eigenvalue decomposition,
Householder preparation and a dense inverse DFT. Verification includes absolute
phase, target-converted states and compiler workspace. Source phase-zero leakage
is retained output and is reported separately from workspace.


In [ ]:
points = []
for precision in PRECISIONS:
    point = verify_program(build_program(precision), precision)
    points.append(point)
    print(point)
assert tuple(point.precision for point in points) == tuple(range(2, 9))


In [ ]:
print("Native squared overlaps:", [point.fidelity for point in points])
print("Native exact CX/U3 depths:", [point.resources.depth for point in points])
print("Native CX counts:", [point.resources.operation_counts["cx"] for point in points])
print("Native widths:", [point.resources.n_qubits for point in points])


In [ ]:
historical_qiskit = {
    "fidelities": HISTORICAL_FIDELITIES,
    "depths": HISTORICAL_DEPTHS,
    "widths": HISTORICAL_WIDTHS,
    "cx_counts": HISTORICAL_CX,
    "remeasured": False,
}
print("Source historical Qiskit observations:", historical_qiskit)


In [ ]:
print(
    "Original optional Qiskit installation cell is entirely commented; no SDK installation is performed"
)


In [ ]:
print("Original optional Qiskit Circuit construction cell is entirely commented")
print("The independent oracle here uses native Circuit, not Qiskit or source imports")


In [ ]:
assert historical_qiskit["remeasured"] is False
print(
    "Original optional Qiskit precision loop is commented; historical arrays stay unchanged"
)


In [ ]:
figure_path = plot_study(tuple(points), output_directory)
display(Image(filename=str(figure_path)))


All original source scales and precisions are retained. Native measured costs
are operator-exact CX/U3 conversion results with existing caller-owned rules;
they are not an optimality guarantee or a performance advantage over the source
historical Qiskit observations. Width annotations denote actual quantum widths.
The whole source workflow includes one physical RHS normalization, documented
above, with its exact input norm and effect on the reported overlap.
